# 震度分布予測 AI — 実験15
Cross-Attention Transformer で震源情報 → 全観測点の計測震度を予測する。

セルを上から順に実行する。

## 1. Google Drive マウント & パス設定

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

# ★ parquet を置いたフォルダのパスに変更してください
DATA_DIR  = Path('/content/drive/MyDrive/seismic_data')
CKPT_DIR  = DATA_DIR / 'checkpoints'
CKPT_DIR.mkdir(parents=True, exist_ok=True)

EQ_PATH  = DATA_DIR / 'earthquakes.parquet'
OBS_PATH = DATA_DIR / 'observations.parquet'

assert EQ_PATH.exists(),  f'Not found: {EQ_PATH}'
assert OBS_PATH.exists(), f'Not found: {OBS_PATH}'
print('Drive マウント OK')

## 2. パッケージインストール

In [ ]:
import time, math
from collections import Counter
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler
import pandas as pd
import numpy as np

print('torch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'なし（CPU）')

## 3. データ読み込み & 前処理

In [ ]:
eq  = pd.read_parquet(EQ_PATH)
obs = pd.read_parquet(OBS_PATH)

# magnitude が null の地震を除外
eq = eq.dropna(subset=['magnitude', 'hypo_lat', 'hypo_lon', 'depth'])

# 地震単位でランダム分割（シード固定で再現性を確保）
rng = np.random.default_rng(seed=42)
all_ids = eq['event_id'].values.copy()
rng.shuffle(all_ids)

n = len(all_ids)
train_ids = set(all_ids[:int(n * 0.85)])
val_ids   = set(all_ids[int(n * 0.85):int(n * 0.925)])
test_ids  = set(all_ids[int(n * 0.925):])

print(f'train: {len(train_ids):,} 件')
print(f'val  : {len(val_ids):,} 件')
print(f'test : {len(test_ids):,} 件')

# obs に震源情報をマージ
obs_merged = obs.merge(
    eq[['event_id','hypo_lat','hypo_lon','depth','magnitude']],
    on='event_id', how='inner'
).dropna(subset=['obs_lat','obs_lon'])

# ★ station_id → 整数インデックスの対応表（Embedding 用）
all_stations   = sorted(obs_merged['station_id'].unique())
station_to_idx = {sid: i for i, sid in enumerate(all_stations)}
N_STATIONS     = len(station_to_idx)
obs_merged['station_idx'] = obs_merged['station_id'].map(station_to_idx)

print(f'obs (merged): {len(obs_merged):,} 件')
print(f'観測点数: {N_STATIONS:,}')

## 4. モデル定義（Cross-Attention Transformer）

In [ ]:
def make_mlp(in_dim: int, hidden: int, out_dim: int, n_layers: int) -> nn.Sequential:
    layers = [nn.Linear(in_dim, hidden), nn.GELU()]
    for _ in range(n_layers - 2):
        layers += [nn.Linear(hidden, hidden), nn.GELU()]
    layers.append(nn.Linear(hidden, out_dim))
    return nn.Sequential(*layers)


class SeismicModel(nn.Module):
    """
    入力:
      src     : (B, 4)    – [hypo_lat, hypo_lon, depth, magnitude]
      obs_pos : (B, N, 4) – [obs_lat, obs_lon, Δlat, Δlon]
      sta_idx : (B, N)    – 観測点インデックス（Embedding用）
      mask    : (B, N)    – True = padding
    出力:
      pred    : (B, N)    – 計測震度
    """

    def __init__(self, hidden: int = 256, n_heads: int = 4,
                 src_layers: int = 3, sta_layers: int = 2,
                 n_stations: int = 8000):
        super().__init__()
        self.src_enc   = make_mlp(4, hidden, hidden, src_layers)
        self.sta_enc   = make_mlp(4, hidden, hidden, sta_layers)
        self.sta_embed = nn.Embedding(n_stations, hidden)
        self.attn      = nn.MultiheadAttention(hidden, n_heads,
                             batch_first=True, dropout=0.1)
        self.norm      = nn.LayerNorm(hidden)
        self.head      = nn.Linear(hidden, 1)

    def forward(self, src: torch.Tensor, obs_pos: torch.Tensor,
                sta_idx: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        ctx = self.src_enc(src).unsqueeze(1)

        hypo_pos   = src[:, :2].unsqueeze(1).expand_as(obs_pos)
        delta      = obs_pos - hypo_pos
        obs_pos_ex = torch.cat([obs_pos, delta], dim=-1)

        q = self.sta_enc(obs_pos_ex) + self.sta_embed(sta_idx)

        out, _ = self.attn(q, ctx, ctx)
        out    = self.norm(out + q)
        return self.head(out).squeeze(-1)


DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
model  = SeismicModel(hidden=256, n_heads=4, n_stations=N_STATIONS).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'パラメータ数: {n_params:,}')
print(f'デバイス: {DEVICE}')

## 5. 損失関数

---
## 6. 実験15 → 16: 遠方限定打ち切り + Exp12b Warm-start

- 打ち切り点を震央から **d ≥ 200km の遠方のみ** に限定（近傍の誤学習を排除）
- **N_CENS_FAR=500**（実験12b の5倍）で遠方打ち切り信号を強化
- **サンプリング重み（BucketSamplerExp15._w）**:
  - 地理: 1°×1°グリッド内件数で正規化（全地域を均等に）
  - マグニチュード: `2^max(0, M-4)` で緩やかに上重み（M4→×1, M5→×2, M6→×4, M7→×8）
  - 深さ: h≥150km の深発地震を×100 で補強
  - ※発生時期は正規化しない（余震多発期の高震度サンプルを間引かないため）
- 実験12b チェックポイントから **Warm-start**（LR=3e-4）
- 震源座標ジッター **JITTER_DEG=0.15**（h < 150km・訓練時のみ）

In [ ]:
## 実験15 設定
BATCH_SIZE  = 32
N_CENS_FAR  = 500      # 遠方打ち切り点の最大数
D_MIN_KM    = 200.0    # 遠方の定義（km）
LAMBDA_CENS = 0.1      # 打ち切り損失の重み
JITTER_DEG  = 0.15     # 震源座標ジッター（h < 150km のみ・訓練時のみ）
LR_EXP15    = 3e-4     # Warm-start の学習率
EPOCHS_15   = 150
PATIENCE_15 = 20

CKPT_EXP12B = CKPT_DIR / 'best_model.pt'
CKPT_EXP15  = CKPT_DIR / 'exp15'
CKPT_EXP15.mkdir(parents=True, exist_ok=True)

assert CKPT_EXP12B.exists(), f'Exp12b チェックポイントが見つかりません: {CKPT_EXP12B}'
print('設定 OK')

In [ ]:
## 全観測点マスタ（打ち切りサンプリング用）
_sta_master = (obs_merged[['station_idx', 'obs_lat', 'obs_lon']]
               .drop_duplicates('station_idx')
               .sort_values('station_idx'))

ALL_STA_LAT = _sta_master['obs_lat'].values.astype('float32')
ALL_STA_LON = _sta_master['obs_lon'].values.astype('float32')
ALL_STA_IDX = _sta_master['station_idx'].values.astype('int64')

print(f'全観測点数: {len(ALL_STA_IDX):,}')

In [ ]:
HUBER_DELTA = 1.0


def intensity_weight(intensity: torch.Tensor) -> torch.Tensor:
    w = torch.ones_like(intensity)
    w = torch.where(intensity >= 3.0, torch.full_like(w,  3.0), w)
    w = torch.where(intensity >= 5.0, torch.full_like(w, 10.0), w)
    return w


def weighted_huber_loss(pred, target, mask, delta=HUBER_DELTA):
    valid  = ~mask
    p, t   = pred[valid], target[valid]
    w      = intensity_weight(t)
    return (F.huber_loss(p, t, reduction='none', delta=delta) * w).mean()


def censored_weighted_huber_loss(pred_obs, target, obs_mask,
                                  pred_cens, cens_mask,
                                  delta=HUBER_DELTA, lambda_cens=0.1):
    obs_loss = weighted_huber_loss(pred_obs, target, obs_mask, delta)

    valid_cens = ~cens_mask
    if valid_cens.any():
        cens_loss = F.relu(pred_cens[valid_cens] - 0.5).pow(2).mean()
    else:
        cens_loss = torch.tensor(0.0, device=pred_obs.device)

    return obs_loss + lambda_cens * cens_loss

In [ ]:
def haversine_km_vec(lat1: float, lon1: float,
                     lat2: np.ndarray, lon2: np.ndarray) -> np.ndarray:
    R = 6371.0
    dlat = np.radians(lat2 - lat1)
    dlon = np.radians(lon2 - lon1)
    a = (np.sin(dlat / 2) ** 2
         + np.cos(np.radians(lat1)) * np.cos(np.radians(lat2))
         * np.sin(dlon / 2) ** 2)
    return 2 * R * np.arcsin(np.sqrt(np.clip(a, 0.0, 1.0)))


class SeismicDatasetExp15(Dataset):
    def __init__(self, event_ids, obs_df, eq_df,
                 all_sta_lat, all_sta_lon, all_sta_idx,
                 n_cens_far=500, d_min_km=200.0,
                 jitter_deg=0.15, training=True):
        self.n_cens_far  = n_cens_far
        self.d_min_km    = d_min_km
        self.jitter_deg  = jitter_deg
        self.training    = training
        self.all_sta_lat = all_sta_lat
        self.all_sta_lon = all_sta_lon
        self.all_sta_idx = all_sta_idx

        eq_sub  = eq_df[eq_df['event_id'].isin(event_ids)].set_index('event_id')
        obs_sub = obs_df[obs_df['event_id'].isin(event_ids)]

        self.samples = []
        for eid, grp in obs_sub.groupby('event_id'):
            if eid not in eq_sub.index:
                continue
            src = eq_sub.loc[eid]
            lat, lon = float(src['hypo_lat']), float(src['hypo_lon'])
            self.samples.append({
                'hypo_lat'   : lat,
                'hypo_lon'   : lon,
                'depth'      : float(src['depth']),
                'magnitude'  : float(src['magnitude']),
                'obs_pos'    : grp[['obs_lat','obs_lon']].values.astype('float32'),
                'sta_idx'    : grp['station_idx'].values.astype('int64'),
                'intensity'  : grp['intensity'].values.astype('float32'),
                'max_int'    : float(grp['intensity'].max()),
                'obs_sta_set': set(grp['station_idx'].values),
                'grid_cell'  : (int(lat), int(lon)),
            })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        lat, lon = s['hypo_lat'], s['hypo_lon']

        if self.training and s['depth'] < 150:
            lat += float(np.random.uniform(-self.jitter_deg, self.jitter_deg))
            lon += float(np.random.uniform(-self.jitter_deg, self.jitter_deg))

        dists    = haversine_km_vec(lat, lon, self.all_sta_lat, self.all_sta_lon)
        far_mask = (dists >= self.d_min_km) & \
                   ~np.isin(self.all_sta_idx, list(s['obs_sta_set']))
        far_cand = np.where(far_mask)[0]

        if len(far_cand) >= self.n_cens_far:
            chosen = np.random.choice(far_cand, self.n_cens_far, replace=False)
        else:
            chosen = far_cand

        cens_pos = np.column_stack([self.all_sta_lat[chosen],
                                     self.all_sta_lon[chosen]]).astype('float32')
        cens_idx = self.all_sta_idx[chosen].astype('int64')

        return (
            torch.tensor([lat, lon, s['depth'], s['magnitude']], dtype=torch.float32),
            torch.tensor(s['obs_pos'],   dtype=torch.float32),
            torch.tensor(s['sta_idx'],   dtype=torch.long),
            torch.tensor(s['intensity'], dtype=torch.float32),
            torch.tensor(cens_pos,       dtype=torch.float32),
            torch.tensor(cens_idx,       dtype=torch.long),
        )


class BucketSamplerExp15(Sampler):
    """
    観測点数でビン分け + 地理均等化 + マグニチュードの緩やかな上重み。
    重み = (1/地理グリッド件数) × 2^max(0,M-4) × depth_w
    """
    BUCKETS = [(1, 5), (6, 30), (31, 100), (101, 10**6)]

    def __init__(self, dataset, batch_size: int, shuffle: bool = True):
        self.batch_size = batch_size
        cell_counts = Counter(s['grid_cell'] for s in dataset.samples)

        bins = {i: [] for i in range(len(self.BUCKETS))}
        for idx, s in enumerate(dataset.samples):
            n = len(s['intensity'])
            for i, (lo, hi) in enumerate(self.BUCKETS):
                if lo <= n <= hi:
                    bins[i].append(idx)
                    break

        rng = np.random.default_rng()
        self.batches = []
        for b_idxs in bins.values():
            if not b_idxs:
                continue
            w = np.array([self._w(dataset.samples[i], cell_counts)
                          for i in b_idxs], dtype=float)
            w /= w.sum()
            n_draw = max(len(b_idxs), batch_size)
            drawn  = rng.choice(b_idxs, size=n_draw, replace=True, p=w).tolist()
            if shuffle:
                rng.shuffle(drawn)
            for start in range(0, len(drawn) - batch_size + 1, batch_size):
                self.batches.append(drawn[start:start + batch_size])
        if shuffle:
            rng.shuffle(self.batches)

    @staticmethod
    def _w(s: dict, cell_counts: Counter) -> float:
        geo_w = 1.0 / cell_counts[s['grid_cell']]          # 地理の均等化
        mag_w = 2.0 ** max(0.0, s['magnitude'] - 4.0)      # M4→×1, M5→×2, M6→×4, M7→×8
        dep_w = 100.0 if s['depth'] >= 150 else 1.0         # 深発地震の補強
        return geo_w * mag_w * dep_w

    def __iter__(self):
        for batch in self.batches:
            yield batch

    def __len__(self):
        return len(self.batches)


def collate_fn_exp15(batch):
    srcs, obs_list, oidx_list, tgt_list, cens_list, cidx_list = zip(*batch)
    srcs = torch.stack(srcs)
    B    = len(obs_list)

    maxlen_obs = max(o.shape[0] for o in obs_list)
    obs_pad  = torch.zeros(B, maxlen_obs, 2)
    oidx_pad = torch.zeros(B, maxlen_obs, dtype=torch.long)
    tgt_pad  = torch.zeros(B, maxlen_obs)
    obs_mask = torch.ones(B, maxlen_obs, dtype=torch.bool)
    for i, (o, ix, t) in enumerate(zip(obs_list, oidx_list, tgt_list)):
        n = o.shape[0]
        obs_pad[i, :n]  = o;  oidx_pad[i, :n] = ix
        tgt_pad[i, :n]  = t;  obs_mask[i, :n] = False

    maxlen_cens = max((c.shape[0] for c in cens_list), default=1)
    cens_pad  = torch.zeros(B, maxlen_cens, 2)
    cidx_pad  = torch.zeros(B, maxlen_cens, dtype=torch.long)
    cens_mask = torch.ones(B, maxlen_cens, dtype=torch.bool)
    for i, (c, ci) in enumerate(zip(cens_list, cidx_list)):
        n = c.shape[0]
        if n > 0:
            cens_pad[i, :n] = c;  cidx_pad[i, :n] = ci
            cens_mask[i, :n] = False

    return srcs, obs_pad, oidx_pad, tgt_pad, obs_mask, cens_pad, cidx_pad, cens_mask


print('定義完了')

In [ ]:
## DataLoader 構築
train_ds_15 = SeismicDatasetExp15(
    train_ids, obs_merged, eq,
    ALL_STA_LAT, ALL_STA_LON, ALL_STA_IDX,
    n_cens_far=N_CENS_FAR, d_min_km=D_MIN_KM,
    jitter_deg=JITTER_DEG, training=True,
)
val_ds_15 = SeismicDatasetExp15(
    val_ids, obs_merged, eq,
    ALL_STA_LAT, ALL_STA_LON, ALL_STA_IDX,
    n_cens_far=N_CENS_FAR, d_min_km=D_MIN_KM,
    jitter_deg=JITTER_DEG, training=False,
)

train_sampler_15 = BucketSamplerExp15(train_ds_15, BATCH_SIZE, shuffle=True)
train_loader_15 = DataLoader(
    train_ds_15, batch_sampler=train_sampler_15,
    collate_fn=collate_fn_exp15, num_workers=2, pin_memory=True,
)
val_loader_15 = DataLoader(
    val_ds_15, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=collate_fn_exp15, num_workers=2, pin_memory=True,
)

print(f'train: {len(train_ds_15):,} events / {len(train_loader_15):,} batches')
print(f'val  : {len(val_ds_15):,} events / {len(val_loader_15):,} batches')

In [ ]:
## 評価関数
@torch.no_grad()
def evaluate_exp15(loader) -> dict:
    model.eval()
    total_loss, total_mae, total_n = 0.0, 0.0, 0
    for src, obs_pos, oidx, tgt, obs_mask, *_ in loader:
        src, obs_pos = src.to(DEVICE), obs_pos.to(DEVICE)
        oidx     = oidx.to(DEVICE)
        tgt      = tgt.to(DEVICE)
        obs_mask = obs_mask.to(DEVICE)
        pred     = model(src, obs_pos, oidx, obs_mask)
        valid    = ~obs_mask
        n        = valid.sum().item()
        if n == 0:
            continue
        total_loss += weighted_huber_loss(pred, tgt, obs_mask).item() * n
        total_mae  += (pred[valid] - tgt[valid]).abs().mean().item() * n
        total_n    += n
    if total_n == 0:
        return {'loss': float('nan'), 'mae': float('nan')}
    return {'loss': total_loss / total_n, 'mae': total_mae / total_n}

In [ ]:
## Warm-start: Exp12b チェックポイント読み込み & Optimizer 設定
ckpt_12b = torch.load(CKPT_EXP12B, map_location=DEVICE)
model.load_state_dict(ckpt_12b['model'])
print(f'Exp12b 読み込み完了 (val_mae={ckpt_12b["val_mae"]:.4f})')

embed_params = list(model.sta_embed.parameters())
other_params = [p for n, p in model.named_parameters() if 'sta_embed' not in n]

optimizer_15 = torch.optim.AdamW([
    {'params': other_params, 'weight_decay': 1e-4},
    {'params': embed_params, 'weight_decay': 1e-2},
], lr=LR_EXP15)
scheduler_15 = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_15, T_max=EPOCHS_15)

print(f'LR={LR_EXP15} で Fine-tuning 準備完了')

In [ ]:
## 学習ループ
history_15      = {'train_loss': [], 'val_loss': [], 'val_mae': []}
best_val_mae_15 = float('inf')
patience_cnt_15 = 0

for epoch in range(1, EPOCHS_15 + 1):
    model.train()
    t0 = time.time()
    total_loss, total_n = 0.0, 0

    # BucketSampler をエポックごとに再生成（シャッフル）
    _sampler    = BucketSamplerExp15(train_ds_15, BATCH_SIZE, shuffle=True)
    _loader     = DataLoader(train_ds_15, batch_sampler=_sampler,
                             collate_fn=collate_fn_exp15, num_workers=2, pin_memory=True)

    for src, obs_pos, oidx, tgt, obs_mask, cens_pos, cidx, cens_mask in _loader:
        src       = src.to(DEVICE);  obs_pos   = obs_pos.to(DEVICE)
        oidx      = oidx.to(DEVICE); tgt       = tgt.to(DEVICE)
        obs_mask  = obs_mask.to(DEVICE)
        cens_pos  = cens_pos.to(DEVICE); cidx  = cidx.to(DEVICE)
        cens_mask = cens_mask.to(DEVICE)

        optimizer_15.zero_grad()
        pred_obs  = model(src, obs_pos,  oidx, obs_mask)
        pred_cens = model(src, cens_pos, cidx, cens_mask)
        loss = censored_weighted_huber_loss(
            pred_obs, tgt, obs_mask, pred_cens, cens_mask,
            lambda_cens=LAMBDA_CENS,
        )
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer_15.step()

        n            = (~obs_mask).sum().item()
        total_loss  += loss.item() * n
        total_n     += n

    scheduler_15.step()
    train_loss  = total_loss / total_n if total_n > 0 else float('nan')
    val_metrics = evaluate_exp15(val_loader_15)
    elapsed     = time.time() - t0

    history_15['train_loss'].append(train_loss)
    history_15['val_loss'].append(val_metrics['loss'])
    history_15['val_mae'].append(val_metrics['mae'])

    print(f'Epoch {epoch:3d}/{EPOCHS_15} '
          f'| train={train_loss:.4f} '
          f'| val_loss={val_metrics["loss"]:.4f} '
          f'| val_mae={val_metrics["mae"]:.4f} '
          f'| {elapsed:.0f}s')

    ckpt = {'epoch': epoch, 'model': model.state_dict(),
            'optimizer': optimizer_15.state_dict(),
            'val_mae': val_metrics['mae'], 'history': history_15}
    torch.save(ckpt, CKPT_EXP15 / f'epoch_{epoch:03d}.pt')

    if not math.isnan(val_metrics['mae']) and val_metrics['mae'] < best_val_mae_15:
        best_val_mae_15 = val_metrics['mae']
        torch.save(ckpt, CKPT_EXP15 / 'best_model.pt')
        patience_cnt_15 = 0
        print(f'  → best (val_mae={best_val_mae_15:.4f})')
    else:
        patience_cnt_15 += 1
        if patience_cnt_15 >= PATIENCE_15:
            print(f'Early stopping at epoch {epoch}')
            break

print(f'\n学習完了。best val_mae = {best_val_mae_15:.4f}')

In [ ]:
## 実験15 誤差分析（震度帯別 MAE）
test_ds_15 = SeismicDatasetExp15(
    test_ids, obs_merged, eq,
    ALL_STA_LAT, ALL_STA_LON, ALL_STA_IDX,
    n_cens_far=N_CENS_FAR, d_min_km=D_MIN_KM,
    jitter_deg=JITTER_DEG, training=False,
)
test_loader_15 = DataLoader(
    test_ds_15, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=collate_fn_exp15, num_workers=2,
)

best_ckpt_15 = torch.load(CKPT_EXP15 / 'best_model.pt', map_location=DEVICE)
model.load_state_dict(best_ckpt_15['model'])
model.eval()

all_pred, all_tgt = [], []
with torch.no_grad():
    for src, obs_pos, oidx, tgt, obs_mask, *_ in test_loader_15:
        src      = src.to(DEVICE)
        obs_pos  = obs_pos.to(DEVICE)
        oidx     = oidx.to(DEVICE)
        tgt      = tgt.to(DEVICE)
        obs_mask = obs_mask.to(DEVICE)
        pred     = model(src, obs_pos, oidx, obs_mask)
        valid    = ~obs_mask
        all_pred.append(pred[valid].cpu())
        all_tgt.append(tgt[valid].cpu())

all_pred = torch.cat(all_pred).numpy()
all_tgt  = torch.cat(all_tgt).numpy()
mae_total = np.abs(all_pred - all_tgt).mean()
print(f'Test MAE (全体): {mae_total:.4f}')

print('\n震度帯別 MAE:')
for lo, hi, label in [(0.5,1.5,'0.5-1.4'), (1.5,2.5,'1.5-2.4'),
                       (2.5,3.5,'2.5-3.4'), (3.5,5.0,'3.5-4.9'),
                       (5.0,9.0,'5.0+')]:
    mask_b = (all_tgt >= lo) & (all_tgt < hi)
    if mask_b.sum() > 0:
        mae_b = np.abs(all_pred[mask_b] - all_tgt[mask_b]).mean()
        print(f'  {label}: MAE={mae_b:.4f}  (n={mask_b.sum():,})')